In [1]:
import pandas as pd
import numpy as np

# Define options
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform', 'uneven']

# Initialize an empty list to store dataframes
results_list = []

# Loop through parameter combinations
for num_warehouses in num_warehouses_options:
    for num_customers in num_customers_options:
        for capacity_distribution in capacity_distribution_options:
            # Construct the filename
            filename = f'results/full_tables/full_results_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv'
            read_df = pd.read_csv(filename)

            # export_results.ipynb appends to these CSVs (mode='a') on every run, so a
            # policy that got re-run (e.g. while debugging one policy at a time) has
            # multiple rows for the same config. Keep only the last one per policy - the
            # most recent run - since append order == chronological order.
            read_df = read_df.drop_duplicates(subset='Policy', keep='last').reset_index(drop=True)

            # Identify the perfect hindsight reward
            perfect_hindsight_reward = read_df.loc[read_df['Policy'] == 'perfect_hindsight', 'MeanReward'].values[0]

            # Create a new dataframe
            new_df = pd.DataFrame()
            new_df['num_warehouses'] = [num_warehouses] * len(read_df)
            new_df['num_customers'] = [num_customers] * len(read_df)
            new_df['capacity_distribution'] = [capacity_distribution] * len(read_df)

            # Add policy columns
            new_df['policy'] = read_df['Policy']
            new_df['avg_reward'] = read_df['MeanReward']
            new_df['std_reward'] = read_df['StdReward']
            new_df['mean_time'] = read_df['MeanTime']
            new_df['std_time'] = read_df['StdTime']
            new_df['all_times'] = read_df['AllTimes'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_rewards'] = read_df['AllRewards'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_distance_ranks'] = read_df['AllFacilityProximity'].apply(lambda x: eval(x) if isinstance(x, str) else x)

            #Get All_times single list
            new_df['all_times_single'] = new_df['all_times'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])

            #Get All_Distance_ranks single list
            new_df['all_distance_ranks_single'] = new_df['all_distance_ranks'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])

            # Compute the percentage above perfect hindsight
            new_df['percentage_above_hindsight'] = (
                (new_df['avg_reward'] - perfect_hindsight_reward) / perfect_hindsight_reward
            ) * 100

            # Propagate std_reward onto the same percentage scale (std doesn't shift with
            # the subtraction, only rescales by the perfect-hindsight-reward factor)
            new_df['percentage_above_hindsight_std'] = (
                new_df['std_reward'] / abs(perfect_hindsight_reward)
            ) * 100

            # Ensure perfect hindsight has 0%
            new_df.loc[new_df['policy'] == 'perfect_hindsight', 'percentage_above_hindsight'] = 0
            new_df.loc[new_df['policy'] == 'perfect_hindsight', 'percentage_above_hindsight_std'] = 0

            # Append to list
            results_list.append(new_df)

# Concatenate all dataframes into a single DataFrame
results = pd.concat(results_list, ignore_index=True)

# Display the results
#print(results.head(100))

In [2]:
num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']


# Define the LaTeX table structure

policies = ['imitation_learning', 'myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','linear_programming_exact','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Percentage Gap to Perfect Hindsight Solution).}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lspi}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{PPO}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
           
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the percentage above perfect hindsight for each policy
            percentages = []
            for policy in policies:
                #print('val', df_filtered.loc[df_filtered['policy'] == policy, 'percentage_above_hindsight'].values)
                #print('Evaluating policy:', policy)
                percentage = df_filtered.loc[df_filtered['policy'] == policy, 'percentage_above_hindsight'].values[0]
                percentages.append(percentage)
            
            # Find the minimum percentage
            min_percentage = min(percentages)
            
            # Format the percentages, highlighting the minimum in bold
            formatted_percentages = [
                f"\\textbf{{{percentage:.2f}}}" if percentage == min_percentage else f"{percentage:.2f}"
                for percentage in percentages
            ]
            
            latex_table += " & ".join(formatted_percentages) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\hline\n"


all_min_perc = []
all_mean_perc = []
all_max_perc = []

for policy in policies:
    all_min_perc.append(results.loc[results['policy'] == policy, 'percentage_above_hindsight'].min())
    all_mean_perc.append(results.loc[results['policy'] == policy, 'percentage_above_hindsight'].mean())
    all_max_perc.append(results.loc[results['policy'] == policy, 'percentage_above_hindsight'].max())


latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy in policies:
    mean_value = all_mean_perc[policies.index(policy)]
    latex_table += f"& \\textbf{{{mean_value:.2f}}}" if mean_value == min(all_mean_perc) else f"& {mean_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy in policies:
    min_value = all_min_perc[policies.index(policy)]
    latex_table += f"& \\textbf{{{min_value:.2f}}}" if min_value == min(all_min_perc) else f"& {min_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy in policies:
    max_value = all_max_perc[policies.index(policy)]
    latex_table += f"& \\textbf{{{max_value:.2f}}}" if max_value == min(all_max_perc) else f"& {max_value:.2f} "
latex_table += r"\\ \bottomrule"


# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Percentage Gap to Perfect Hindsight Solution).}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lspi}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf

<>:76: SyntaxWarning: invalid escape sequence '\m'
<>:81: SyntaxWarning: invalid escape sequence '\m'
<>:86: SyntaxWarning: invalid escape sequence '\m'
<>:76: SyntaxWarning: invalid escape sequence '\m'
<>:81: SyntaxWarning: invalid escape sequence '\m'
<>:86: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/3511495729.py:76: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/3511495729.py:81: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/3511495729.py:86: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"


In [3]:
num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']


# Define the LaTeX table structure

policies = ['imitation_learning', 'myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','linear_programming_exact','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{landscape}
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Percentage Gap to Perfect Hindsight Solution).}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|c|cc|cc|cc|cccc}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lspi}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{PPO}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
           
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the percentage above perfect hindsight (and its std) for each policy
            percentages = []
            stds = []
            for policy in policies:
                #print('val', df_filtered.loc[df_filtered['policy'] == policy, 'percentage_above_hindsight'].values)
                #print('Evaluating policy:', policy)
                percentage = df_filtered.loc[df_filtered['policy'] == policy, 'percentage_above_hindsight'].values[0]
                std = df_filtered.loc[df_filtered['policy'] == policy, 'percentage_above_hindsight_std'].values[0]
                percentages.append(percentage)
                stds.append(std)
            
            # Find the minimum percentage
            min_percentage = min(percentages)
            
            # Format the percentages, highlighting the minimum in bold; std shown in
            # parentheses before the mean
            formatted_percentages = [
                f"\\textbf{{{percentage:.2f}}} ({std:.2f})" if percentage == min_percentage else f"{percentage:.2f} ({std:.2f})"
                for percentage, std in zip(percentages, stds)
            ]
            
            latex_table += " & ".join(formatted_percentages) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\hline\n"


all_min_perc = []
all_mean_perc = []
all_max_perc = []

for policy in policies:
    all_min_perc.append(results.loc[results['policy'] == policy, 'percentage_above_hindsight'].min())
    all_mean_perc.append(results.loc[results['policy'] == policy, 'percentage_above_hindsight'].mean())
    all_max_perc.append(results.loc[results['policy'] == policy, 'percentage_above_hindsight'].max())


latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy in policies:
    mean_value = all_mean_perc[policies.index(policy)]
    latex_table += f"& \\textbf{{{mean_value:.2f}}}" if mean_value == min(all_mean_perc) else f"& {mean_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy in policies:
    min_value = all_min_perc[policies.index(policy)]
    latex_table += f"& \\textbf{{{min_value:.2f}}}" if min_value == min(all_min_perc) else f"& {min_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy in policies:
    max_value = all_max_perc[policies.index(policy)]
    latex_table += f"& \\textbf{{{max_value:.2f}}}" if max_value == min(all_max_perc) else f"& {max_value:.2f} "
latex_table += r"\\ \bottomrule"


# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
\end{landscape}
"""

# Print the LaTeX table string
print(latex_table)


\begin{landscape}
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Percentage Gap to Perfect Hindsight Solution).}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|c|cc|cc|cc|cccc}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lspi}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{PPO}\\
        \toprule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 2.57 (7.87

<>:81: SyntaxWarning: invalid escape sequence '\m'
<>:86: SyntaxWarning: invalid escape sequence '\m'
<>:91: SyntaxWarning: invalid escape sequence '\m'
<>:81: SyntaxWarning: invalid escape sequence '\m'
<>:86: SyntaxWarning: invalid escape sequence '\m'
<>:91: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/2595029407.py:81: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/2595029407.py:86: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/2595029407.py:91: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"


In [4]:
num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']


policies = ['imitation_learning', 'myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','linear_programming_exact','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Online Runtime per Order in milliseconds).}
    \label{tab:policy_comparison_time}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lspi}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the percentage above perfect hindsight for each policy
            times = []
            short_times = []
            for policy in policies:
                time = df_filtered.loc[df_filtered['policy'] == policy, 'mean_time'].values[0]
                time = time / 1_000
                if policy not in ['myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']:
                    time = time / 1_000  # Convert to milliseconds
                else:
                    short_times.append(time)
                times.append(time)

            # Find the min time in times, knowing that short_times contains the relevant ones
            min_time = min(short_times)

            # Format the percentages, highlighting the minimum in bold
            formatted_times = [
                f"\\textbf{{{time:.2f}}}" if time == min_time else f"{time:.2f}"
                for time in times
            ]
            
            latex_table += " & ".join(formatted_times) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Add average, min, and max rows
all_min_time = []
all_mean_time = []
all_max_time = []   
for policy in policies:
    all_min_time.append(results.loc[results['policy'] == policy, 'mean_time'].min())
    all_mean_time.append(results.loc[results['policy'] == policy, 'mean_time'].mean())
    all_max_time.append(results.loc[results['policy'] == policy, 'mean_time'].max())

short_time_policies = ['myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def convert_time(value, policy):
    value = value / 1_000
    if policy not in short_time_policies:
        value = value / 1_000  # Convert to milliseconds
    return value

converted_mean_time = [convert_time(v, p) for v, p in zip(all_mean_time, policies)]
converted_min_time = [convert_time(v, p) for v, p in zip(all_min_time, policies)]
converted_max_time = [convert_time(v, p) for v, p in zip(all_max_time, policies)]

best_mean_short = min(v for v, p in zip(converted_mean_time, policies) if p in short_time_policies)
best_mean_long = min(v for v, p in zip(converted_mean_time, policies) if p not in short_time_policies)
best_min_short = min(v for v, p in zip(converted_min_time, policies) if p in short_time_policies)
best_min_long = min(v for v, p in zip(converted_min_time, policies) if p not in short_time_policies)
best_max_short = min(v for v, p in zip(converted_max_time, policies) if p in short_time_policies)
best_max_long = min(v for v, p in zip(converted_max_time, policies) if p not in short_time_policies)

latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy, mean_value in zip(policies, converted_mean_time):
    latex_table += f"& \\textbf{{{mean_value:.2f}}}" if mean_value == best_mean_short else f"& {mean_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy, min_value in zip(policies, converted_min_time):
    latex_table += f"& \\textbf{{{min_value:.2f}}}" if min_value == best_min_short else f"& {min_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy, max_value in zip(policies, converted_max_time):
    latex_table += f"& \\textbf{{{max_value:.2f}}}" if max_value == best_max_short else f"& {max_value:.2f} "
latex_table += r"\\ \bottomrule"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
    \par\centering\tiny{Columns marked with * correspond to very low runtime values and are multiplied by $10^3$. Consequently, the values in these columns can be directly interpreted as microseconds.}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Online Runtime per Order in milliseconds).}
    \label{tab:policy_comparison_time}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lspi}\textsu

<>:94: SyntaxWarning: invalid escape sequence '\m'
<>:98: SyntaxWarning: invalid escape sequence '\m'
<>:102: SyntaxWarning: invalid escape sequence '\m'
<>:94: SyntaxWarning: invalid escape sequence '\m'
<>:98: SyntaxWarning: invalid escape sequence '\m'
<>:102: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/2737780408.py:94: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/2737780408.py:98: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/2737780408.py:102: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"


In [5]:
num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']


policies = ['imitation_learning', 'myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','linear_programming_exact','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{landscape}
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Online Runtime per Order in milliseconds, Mean (Std)).}
    \label{tab:policy_comparison_time_std}
    \begin{tabular}{ccc|c|cc|cc|cc|cccc}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lspi}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the mean (and std) runtime for each policy
            times = []
            stds = []
            short_times = []
            for policy in policies:
                time = df_filtered.loc[df_filtered['policy'] == policy, 'mean_time'].values[0]
                std = df_filtered.loc[df_filtered['policy'] == policy, 'std_time'].values[0]
                if policy in ['myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']:
                    time = time / 1_000  # Convert to microseconds
                    std = std / 1_000
                    short_times.append(time)
                else:
                    time = time / 1_000_000  # Convert to milliseconds
                    std = std / 1_000_000

                times.append(time)
                stds.append(std)

            # Find the min time in times, knowing that short_times contains the relevant ones
            min_time = min(short_times)

            # Format the times, highlighting the minimum in bold; std shown in parentheses after the mean
            formatted_times = [
                f"\\textbf{{{time:.2f}}} ({std:.2f})" if time == min_time else f"{time:.2f} ({std:.2f})"
                for time, std in zip(times, stds)
            ]
            
            latex_table += " & ".join(formatted_times) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Add average, min, and max rows
all_min_time = []
all_mean_time = []
all_max_time = []   
for policy in policies:
    all_min_time.append(results.loc[results['policy'] == policy, 'mean_time'].min())
    all_mean_time.append(results.loc[results['policy'] == policy, 'mean_time'].mean())
    all_max_time.append(results.loc[results['policy'] == policy, 'mean_time'].max())

short_time_policies = ['myopic', 'genetic_programming', 'least_squares_policy_iteration', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def convert_time(value, policy):
    if policy in short_time_policies:
        return value / 1_000  # Convert to microseconds
    return value / 1_000_000  # Convert to milliseconds

converted_mean_time = [convert_time(v, p) for v, p in zip(all_mean_time, policies)]
converted_min_time = [convert_time(v, p) for v, p in zip(all_min_time, policies)]
converted_max_time = [convert_time(v, p) for v, p in zip(all_max_time, policies)]

best_mean_short = min(v for v, p in zip(converted_mean_time, policies) if p in short_time_policies)
best_mean_long = min(v for v, p in zip(converted_mean_time, policies) if p not in short_time_policies)
best_min_short = min(v for v, p in zip(converted_min_time, policies) if p in short_time_policies)
best_min_long = min(v for v, p in zip(converted_min_time, policies) if p not in short_time_policies)
best_max_short = min(v for v, p in zip(converted_max_time, policies) if p in short_time_policies)
best_max_long = min(v for v, p in zip(converted_max_time, policies) if p not in short_time_policies)

latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy, mean_value in zip(policies, converted_mean_time):
    latex_table += f"& \\textbf{{{mean_value:.2f}}}" if mean_value == best_mean_short else f"& {mean_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy, min_value in zip(policies, converted_min_time):
    latex_table += f"& \\textbf{{{min_value:.2f}}}" if min_value == best_min_short else f"& {min_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy, max_value in zip(policies, converted_max_time):
    latex_table += f"& \\textbf{{{max_value:.2f}}}" if max_value == best_max_short else f"& {max_value:.2f} "
latex_table += r"\\ \bottomrule"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
    \par\centering\tiny{Columns marked with * correspond to very low runtime values and are multiplied by $10^3$. Consequently, the values in these columns can be directly interpreted as microseconds.}
\end{table}
\end{landscape}
"""

# Print the LaTeX table string
print(latex_table)


<>:100: SyntaxWarning: invalid escape sequence '\m'
<>:104: SyntaxWarning: invalid escape sequence '\m'
<>:108: SyntaxWarning: invalid escape sequence '\m'
<>:100: SyntaxWarning: invalid escape sequence '\m'
<>:104: SyntaxWarning: invalid escape sequence '\m'
<>:108: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/3791818496.py:100: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/3791818496.py:104: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_9048/3791818496.py:108: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"



\begin{landscape}
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy Comparison for Different Instances (Online Runtime per Order in milliseconds, Mean (Std)).}
    \label{tab:policy_comparison_time_std}
    \begin{tabular}{ccc|c|cc|cc|cc|cccc}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{9}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{4}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lspi}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{lpe}} & \textbf{\gls{pla}} & \te

In [6]:
import pandas as pd

# Read the theta.csv file
theta_df = pd.read_csv('training/least_squares_policy_iteration_training/theta.csv')

# Define the LaTeX table structure
latex_table = r"""
\begin{table}[H]
    \centering
    \caption{\gls{lspi} Final Weights ($w$) for Different Instances.}
    \label{tab:lspi_w}
    \begin{tabular}{cS[table-format=4.2]S[table-format=3.2]S[table-format=2.2]}
        \toprule
        \textbf{Facilities $|\FacilitySet|$} & \textbf{$w_1$} & \textbf{$w_2$} & \textbf{$w_3$} \\
        \midrule
"""

# Define the unique combinations of num_customers, num_warehouses, and capacity_distribution
unique_combinations = theta_df[['num_customers', 'num_warehouses', 'capacity_distribution']].drop_duplicates()

num_customers = 50
capacity_distribution = 'uniform'

# Loop through the unique combinations to fill the table
for num_warehouses in unique_combinations['num_warehouses'].unique():
        latex_table += f"{num_warehouses} & "
        
        # Filter the DataFrame for the current configuration
        df_filtered = theta_df[
            (theta_df["num_customers"] == num_customers) & 
            (theta_df["num_warehouses"] == num_warehouses) & 
            (theta_df["capacity_distribution"] == capacity_distribution)
        ]
        
        # Extract the theta values
        if not df_filtered.empty:
            theta = df_filtered.iloc[0]['theta']
            theta1 = round(float(theta.split(',')[0]), 2)
            theta2 = round(float(theta.split(',')[1]), 2)
            theta3 = round(float(theta.split(',')[2]), 2)
            latex_table += f"{theta1} & {theta2} & {theta3} \\\\\n"
        else:
            latex_table += " & \\\\\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \centering
    \caption{\gls{lspi} Final Weights ($w$) for Different Instances.}
    \label{tab:lspi_w}
    \begin{tabular}{cS[table-format=4.2]S[table-format=3.2]S[table-format=2.2]}
        \toprule
        \textbf{Facilities $|\FacilitySet|$} & \textbf{$w_1$} & \textbf{$w_2$} & \textbf{$w_3$} \\
        \midrule
2 & -2542.52 & -167.81 & 83.8 \\
3 & -2012.1 & -128.04 & 95.93 \\
4 & -1597.72 & -47.7 & 40.97 \\
5 & -1524.81 & -56.11 & 39.78 \\
        \bottomrule

    \end{tabular}
\end{table}



In [10]:
import pandas as pd

# Read the theta.csv file
theta_df = pd.read_csv('training/least_squares_policy_iteration_training/theta.csv')

num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']

num_customers_options = [50, 100]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']

# Define the LaTeX table structure
latex_table = r"""
\begin{table}[H]
    \centering
    \caption{\gls{lspi} Final Weights ($w$) for All Instances.}
    \label{tab:lspi_w_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$w_1$} & \textbf{$w_2$} & \textbf{$w_3$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = theta_df[
                (theta_df["num_customers"] == num_customers) &
                (theta_df["num_warehouses"] == num_warehouses) &
                (theta_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the theta values
            if not df_filtered.empty:
                theta = df_filtered.iloc[0]['theta']
                theta1 = round(float(theta.split(',')[0]), 2)
                theta2 = round(float(theta.split(',')[1]), 2)
                theta3 = round(float(theta.split(',')[2]), 2)
                latex_table += f"{theta1} & {theta2} & {theta3} \\\\\n"
            else:
                latex_table += " & & \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \centering
    \caption{\gls{lspi} Final Weights ($w$) for All Instances.}
    \label{tab:lspi_w_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$w_1$} & \textbf{$w_2$} & \textbf{$w_3$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & -2542.52 & -167.81 & 83.8 \\
& & Uneven & -2743.71 & -146.53 & 92.76 \\
& \multirow{2}{*}{3} & Uniform & -2012.1 & -128.04 & 95.93 \\
& & Uneven & -2197.72 & -111.94 & 105.21 \\
& \multirow{2}{*}{4} & Uniform & -1597.72 & -47.7 & 40.97 \\
& & Uneven & -1870.74 & -87.22 & 34.38 \\
& \multirow{2}{*}{5} & Uniform & -1524.81 & -56.11 & 39.78 \\
& & Uneven & -1622.63 & -55.06 & 46.97 \\
        \hline
        \multirow{8}{*}{100} & \multirow{2}{*}{2} & Uniform & -3975.89 & -136.09 & 114.18 \\
& & Uneven & -4167.2 & -73.49 & 115.83 \\
& \multirow{2}{*}{3} & Uniform

In [9]:
# PLA (parameterized lookahead approximation) sensitivity to theta, as a table instead
# of the plot above: per instance, the mean reward at each tested theta plus the
# selected (best) theta. Best = highest mean_reward (reward is negative distance, so
# less negative is better - see Eq. 9/23 as in the plot cell above).
import pandas as pd

num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']
param_options = [1.0, 1.02, 1.04, 1.06, 1.08, 1.10]

pla_df = pd.read_csv('training/parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_all_params.csv')

latex_table = r"""
\begin{landscape}
\begin{table}[H]
    \tiny
    \centering
    \caption{\gls{pla} Mean Reward per Tested $\theta$, and Selected $\theta$, for All Instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean reward per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = pla_df[
                (pla_df["num_customers"] == num_customers) &
                (pla_df["num_warehouses"] == num_warehouses) &
                (pla_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the mean reward for each tested theta
            rewards = []
            for param in param_options:
                reward = df_filtered.loc[df_filtered['param'] == param, 'mean_reward'].values[0]
                rewards.append(reward)

            # Best theta = highest mean reward (least negative)
            best_reward = max(rewards)
            best_param = param_options[rewards.index(best_reward)]

            formatted_rewards = [
                f"\\textbf{{{reward:.2f}}}" if reward == best_reward else f"{reward:.2f}"
                for reward in rewards
            ]

            latex_table += " & ".join(formatted_rewards) + f" & {best_param:.2f} " + r" \\" + "\n"
    latex_table += "        \\hline\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
\end{landscape}
"""

# Print the LaTeX table string
print(latex_table)



\begin{landscape}
\begin{table}[H]
    \tiny
    \centering
    \caption{\gls{pla} Mean Reward per Tested $\theta$, and Selected $\theta$, for All Instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean reward per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & -3306.03 & -3295.14 & -3295.98 & -3289.75 & -3296.60 & \textbf{-3287.04} & 1.10  \\
& & Uneven & -3507.93 & -3496.16 & -3493.34 & -3492.69 & \textbf{-3490.43} & -3496.31 